In [ ]:
#| default_exp exhash

In [ ]:
#| export
from fastcore.utils import *
from dialoghelper.core import *
from exhash import *

In [ ]:
msg = await read_msg()
id_ = msg.id
cts = msg.content

In [ ]:
#| export
async def msg_lnhashview(
    id:str, # id of message to view
    dname:str='', # Dialog containing message; defaults to current dialog
):
    "Show lnhash-addressed lines of a message"
    msg = await read_msgid(id=id, dname=dname)
    if not msg: return 'No such message.'
    return '\n'.join(lnhashview(msg['content']))

In [ ]:
#| export
async def msg_exhash(
    id:str, # id of message to edit
    cmds:list, # List of exhash command strings to apply
    dname:str='', # Dialog containing message; defaults to current dialog
):
    """Verified line-addressed editor. Apply commands to msg `id` contents, return lnhash diff.
    **NB**: *all* exhash commands *must* start with an address.
    The *only* allowed addresses are a single lnhash, or a pair separated by `,`. (I.e no `%`, `.`, etc.)
    **NB**: hashes are checked before each command is run. So be sure to have commands go last->first order to avoid changing earlier lines.

    Commands are like `ex`, but use lnhash addresses instead of bare line numbers: ``lineno|hash|cmd`` where hash is a 4-char hex content hash.
    Use `msg_lnhashview(text)` to get addresses before first use.

    Addressing:
        Single:   ``12|a3f2|cmd``
        Range:    ``12|a3f2|,15|b1c3|cmd``
        Special:  ``0|0000|`` targets before line 1 (only with a or i)

    Commands:
        s/pat/rep/[flags]  Substitute (regex). Flags: g=all, i=case-insensitive
        d                  Delete line(s)
        a                  Append text after line
        i                  Insert text before line
        c                  Change/replace line(s)
        j                  Join with next line; with range, joins all
        m dest             Move line(s) after dest address
        t dest             Copy line(s) after dest address
        >[n]               Indent n levels (default 1, 4 spaces each)
        <[n]               Dedent n levels (default 1)
        sort               Sort lines alphabetically
        p                  Print (include in output without changing)
        g/pat/cmd          Global: run cmd on matching lines
        g!/pat/cmd         Inverted global (also v/pat/cmd)

    `cmds` is a required list of command strings. For `a`/`i`/`c`, include the text block in the same command string after a newline.
    Unlike `ex`, do NOT add a trailing `.` line — the string boundary ends the block. A trailing `.` will be inserted literally (with a warning)."""
    msg = await read_msgid(id=id, dname=dname)
    if not msg: return 'No such message.'
    txt = msg['content']
    edit = exhash(txt, cmds)
    res = '\n'.join(edit['lines'])
    upres = await update_msg(id=id, content=res, dname=dname)
    assert upres.startswith('_'), f"Message update failed: {upres}"
    return edit.format_diff()